[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q2/02_load_and_filter.ipynb)

# R2-Q2 Week 2 — Load R2-Q1's outputs and filter to your working set

This notebook turns your Week-1 commitments into a concrete working data set. It loads four R2-Q1 outputs — the PlantDoc prediction table (`pd_predictions.parquet`), R2-Q1's class-space mappings (from `precommit.json`), the PV-internal predictions table for reference, and the path to the trained ResNet-18 classifier (N03 loads the weights) — then filters PD predictions to the cases where the model got the disease category wrong and runs exploratory analysis on the resulting set. The Week 2 deliverable is `working_set.parquet`: the annotated misclassification table that N03 (Grad-CAM and sanity checks) and N04 (taxonomy application) both consume.

By the end of this notebook you will have:

- A `working_set.parquet` file in your output directory — about 80 rows, each a PlantDoc image the PlantVillage-trained model misclassified at the disease-category level. Each row carries the PD class label, the PV-class prediction, both class spaces' category labels, the model's confidence, the PD host and disease metadata, and the PV-class integer index N03's Grad-CAM step needs.
- An exploratory picture of the misclassification set: category-level distribution (the basis for N04's eventual taxonomy distribution), per-host and per-class breakdowns (which your aggregation-level pre-commit in N01 already marked as hypothesis-only), and the model-confidence distribution among the wrong predictions.
- Figures from those analyses, sized and styled to drop into the Week-4 presentation.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in the following Cell:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in the following Cell:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

In [ ]:
# Mount Drive and set up irilab2026.
import irilab2026 as iri
iri.setup(gpu_required=False)

OUTPUT_DIR = iri.output_dir("r2_q2")
print(f"Output directory: {OUTPUT_DIR}")

## 1) Load the N01 pre-commit

`precommit.json` is the four-field commitment file you wrote in Notebook 01 — your taxonomy, your categorization procedure, your aggregation level, and your sanity-check criterion. The rest of this notebook doesn't read individual fields from it (those get used in Notebooks 03 and 04); what matters here is that the file exists and is well-formed. If it doesn't exist, the most likely reason is that Notebook 01 wasn't run, and the error below says so.

In [ ]:
### 1.1) Defensive load of precommit.json

import json

try:
    with open(OUTPUT_DIR / "precommit.json") as f:
        precommit = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(
        "Could not find precommit.json in OUTPUT_DIR. "
        "This file is produced by 01_orientation_and_precommit.ipynb — "
        "run that notebook first."
    ) from None

print(f"precommit.json loaded from {OUTPUT_DIR / 'precommit.json'}")
print(f"  top-level keys: {sorted(precommit.keys())}")